# RF-DETR

fine tuning rf-detr large to find plastic bags. we train it twice with different seeds and average the weights into one model, and when predicting it also runs every image flipped left to right.

it only trains on the training split, so the val score at the end is on images it never saw

In [ ]:
%pip install -q --no-cache-dir "rfdetr[train]==1.11.0"

In [ ]:
import json, os, shutil, time
from pathlib import Path

import numpy as np
import torch
import torchvision.transforms.functional as TF
from PIL import Image
from torchvision.ops import box_iou

RESOLUTION = 704  # 896 did worse
EPOCHS = 25       # val stops getting better around here
BATCH = 4
GRAD_ACCUM = 4    # 16 images per step
WORKERS = 4       # 0 if you get a "bus error"
SEEDS = [0, 1]    # one run per seed, then we average them

ROOT = Path.cwd().parent  # the notebook is in event_materials/
DATA = ROOT / "data"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = ROOT / "runs" / f"rfdetr_large_{RESOLUTION}"
MODEL_PATH = OUT / "model.pt"  # final model

## Data

In [ ]:
def read_split(split):
    # (image, boxes) for every image with a label file, same as the starter
    folder = DATA / split
    items = []
    for img in sorted((folder / "image").glob("*.jpg")):
        label = folder / "label" / f"{img.stem}.txt"
        if not label.exists():
            continue
        boxes = []
        for line in label.read_text().splitlines():
            p = line.split()
            if len(p) >= 8 and p[0] == "plastic_bag":
                x1, y1, x2, y2 = map(float, p[4:8])
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
        items.append((img, np.array(boxes, np.float32).reshape(-1, 4)))
    return items


def write_coco(items, folder):
    # rf-detr wants coco format, the images + an _annotations.coco.json
    folder.mkdir(parents=True, exist_ok=True)
    images, anns = [], []
    for i, (img, boxes) in enumerate(items, start=1):  # coco ids start at 1
        dst = folder / img.name
        if not dst.exists():
            try:
                os.link(img, dst)  # hardlink so we don't copy the images
            except OSError:
                shutil.copy2(img, dst)
        w, h = Image.open(img).size
        images.append({"id": i, "file_name": img.name, "width": w, "height": h})
        for x1, y1, x2, y2 in boxes.tolist():
            anns.append({"id": len(anns) + 1, "image_id": i, "category_id": 0, "iscrowd": 0,
                         "bbox": [x1, y1, x2 - x1, y2 - y1], "area": (x2 - x1) * (y2 - y1)})
    cats = [{"id": 0, "name": "plastic_bag", "supercategory": "none"}]
    (folder / "_annotations.coco.json").write_text(json.dumps({"images": images, "annotations": anns, "categories": cats}))


train, val = read_split("training"), read_split("val")
DATASET = ROOT / "runs" / "rfdetr_data"
write_coco(train, DATASET / "train")
write_coco(val, DATASET / "valid")
len(train), len(val)

## Train

~16 min per seed on the A10G. skips training if model.pt or both seeds are already there

In [ ]:
from rfdetr import RFDETRLarge
import torch_linear_assignment.assignment as tla

# triton doesn't work on this aws image, use the scipy matcher instead
tla._cuda_uses_triton = lambda cost: False

if MODEL_PATH.exists() or all((OUT / f"seed{s}" / "last_ema.pth").exists() for s in SEEDS):
    print(f"already trained, skipping. delete {OUT} to train again")
else:
    for old in OUT.glob("**/*.pth"):
        old.unlink()  # clear old runs
    for seed in SEEDS:
        start = time.time()
        RFDETRLarge(resolution=RESOLUTION).train(
            dataset_dir=str(DATASET),
            output_dir=str(OUT / f"seed{seed}"),
            epochs=EPOCHS,
            batch_size=BATCH,
            grad_accum_steps=GRAD_ACCUM,
            lr=1e-4,
            lr_encoder=1.5e-4,
            num_workers=WORKERS,
            checkpoint_interval=10**6,  # no extra checkpoints
            tensorboard=False,
            progress_bar="tqdm",
            seed=seed,
            augmentation_backend="torchvision",  # keep the default augs
        )
        print(f"seed {seed} took {(time.time() - start) / 60:.0f} min")
        for f in [*(OUT / f"seed{seed}").glob("*.ckpt"), *(OUT / f"seed{seed}").glob("checkpoint_best_*.pth")]:
            f.unlink()  # only need last_ema, the rest just takes up space
        torch.cuda.empty_cache()

## Scoring

same evaluate_map50 as the starter notebook

In [ ]:
@torch.no_grad()
def evaluate_map50(model, loader, device) -> float:
    model.eval()
    all_scores = []
    all_matches = []
    total_gt = 0
    for images, targets in loader:
        images_d = [img.to(device) for img in images]
        preds = model(images_d)
        for pred, tgt in zip(preds, targets):
            gt_boxes = tgt["boxes"]
            total_gt += int(gt_boxes.shape[0])
            p_boxes = pred["boxes"].detach().cpu()
            p_scores = pred["scores"].detach().cpu()
            order = p_scores.argsort(descending=True)
            p_boxes = p_boxes[order]
            p_scores = p_scores[order]
            matched = torch.zeros(gt_boxes.shape[0], dtype=torch.bool)
            if gt_boxes.numel() == 0:
                for s in p_scores.tolist():
                    all_scores.append(s)
                    all_matches.append(0)
                continue
            if p_boxes.numel() == 0:
                continue
            iou = box_iou(p_boxes, gt_boxes)
            for pi in range(p_boxes.shape[0]):
                ious_p = iou[pi].clone()
                ious_p[matched] = -1.0
                best_iou, best_gt = ious_p.max(dim=0)
                if best_iou.item() >= 0.5:
                    matched[best_gt] = True
                    all_matches.append(1)
                else:
                    all_matches.append(0)
                all_scores.append(float(p_scores[pi].item()))
    if total_gt == 0 or not all_scores:
        return 0.0
    scores_t = torch.tensor(all_scores)
    matches_t = torch.tensor(all_matches, dtype=torch.float32)
    order = scores_t.argsort(descending=True)
    matches_t = matches_t[order]
    tp_cum = torch.cumsum(matches_t, dim=0)
    fp_cum = torch.cumsum(1 - matches_t, dim=0)
    recall = tp_cum / max(1, total_gt)
    precision = tp_cum / (tp_cum + fp_cum).clamp(min=1e-9)
    ap = 0.0
    for t in torch.linspace(0, 1, 11):
        mask = recall >= t
        p = precision[mask].max().item() if mask.any() else 0.0
        ap += p / 11.0
    return float(ap)


def batches(items, size=8):
    # same batches as the starter's test loader
    for i in range(0, len(items), size):
        chunk = items[i:i + size]
        yield ([TF.to_tensor(Image.open(path).convert("RGB")) for path, _ in chunk],
               [{"boxes": torch.as_tensor(boxes)} for _, boxes in chunk])

## The model

wraps rf-detr so it works like the starter's model: a list of image tensors in, boxes and scores out. it also runs the flipped image and keeps the best boxes from both

In [ ]:
from rfdetr import RFDETR
from torchvision.ops import nms


class RFDETRBags(torch.nn.Module):
    def __init__(self, path):
        super().__init__()
        # needs the full path. the checkpoint has the size and classes saved in it
        self.rfdetr = RFDETR.from_checkpoint(str(Path(path).resolve()), device=DEVICE.type)

    @torch.no_grad()
    def forward(self, images, targets=None):
        images = list(images)
        # low threshold so we keep pretty much every box
        dets = self.rfdetr.predict(images, threshold=0.001, include_source_image=False)
        # same thing on the flipped images
        flips = self.rfdetr.predict([img.flip(-1) for img in images], threshold=0.001, include_source_image=False)
        out = []
        for img, d, f in zip(images, dets, flips):
            w = img.shape[-1]
            fx1, fy1, fx2, fy2 = torch.as_tensor(f.xyxy, dtype=torch.float32).reshape(-1, 4).unbind(1)
            boxes = torch.cat([torch.as_tensor(d.xyxy, dtype=torch.float32).reshape(-1, 4),
                               torch.stack([w - fx2, fy1, w - fx1, fy2], dim=1)])  # flip those boxes back
            scores = torch.cat([torch.as_tensor(d.confidence, dtype=torch.float32),
                                torch.as_tensor(f.confidence, dtype=torch.float32)])
            keep = nms(boxes, scores, 0.6)  # drop the duplicates
            out.append({"boxes": boxes[keep], "scores": scores[keep], "labels": torch.ones(len(keep), dtype=torch.long)})
        return out

## Average the seeds

both runs start from the same pretrained weights, so we can average them into one model. if the seed runs aren't here it just uses the saved model.pt

In [ ]:
if all((OUT / f"seed{s}" / "last_ema.pth").exists() for s in SEEDS):
    cks = [torch.load(OUT / f"seed{s}" / "last_ema.pth", map_location="cpu", weights_only=False) for s in SEEDS]
    avg = cks[0]
    for k, v in avg["model"].items():
        if v.is_floating_point():
            avg["model"][k] = sum(c["model"][k] for c in cks) / len(cks)
    for k in ("state_dict", "optimizer_states", "callbacks"):
        avg.pop(k, None)  # only "model" is needed
    torch.save(avg, MODEL_PATH)
    del cks, avg
    for s in SEEDS:  # how each seed does on its own
        print(f"seed {s}: {evaluate_map50(RFDETRBags(OUT / f'seed{s}' / 'last_ema.pth'), batches(val), DEVICE):.4f}")
else:
    print(f"no seed runs here, using the saved {MODEL_PATH}")

## Load it back

In [ ]:
print(f"{MODEL_PATH.stat().st_size / 1e6:.0f} MB")

# load the saved file and score it to make sure it works
model = RFDETRBags(MODEL_PATH)
print("val mAP@50:", evaluate_map50(model, batches(val), DEVICE))

## What it's predicting

green = the real bags, red = what the model found (with its score)

In [ ]:
import matplotlib.patches as patches
import matplotlib.pyplot as plt

# first 2 are the ones the starter shows, the rest are spread across val
picks = val[:2] + [val[i] for i in np.linspace(2, len(val) - 1, 5)[1:].astype(int)]
_, axes = plt.subplots(len(picks), 2, figsize=(12, 5 * len(picks)), squeeze=False)
for (ax_gt, ax_pr), (path, boxes) in zip(axes, picks):
    img = Image.open(path).convert("RGB")
    pred = model([TF.to_tensor(img).to(DEVICE)])[0]
    keep = pred["scores"] >= 0.5
    ax_gt.imshow(img)
    ax_gt.set_title(f"Ground truth: {len(boxes)} ({path.name})")
    ax_pr.imshow(img)
    ax_pr.set_title(f"Predictions (score > 0.5): {int(keep.sum())}")
    for x1, y1, x2, y2 in boxes:
        ax_gt.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2))
    for (x1, y1, x2, y2), s in zip(pred["boxes"][keep].tolist(), pred["scores"][keep].tolist()):
        ax_pr.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linewidth=2))
        ax_pr.text(x1, max(0, y1 - 4), f"{s:.2f}", color="red", fontsize=9, bbox=dict(facecolor="white", alpha=0.6, pad=1))
    ax_gt.axis("off")
    ax_pr.axis("off")
plt.tight_layout()
plt.show()

## Test

In [ ]:
AWS_ACCESS_KEY_ID = ""
AWS_SECRET_ACCESS_KEY = ""


def download_test():
    # same download as the starter, skips files we already have
    import boto3
    s3 = boto3.resource("s3", aws_access_key_id=AWS_ACCESS_KEY_ID, aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
                        region_name="us-east-1")
    bucket = s3.Bucket("uprm-hackathon-data")
    for obj in bucket.objects.filter(Prefix="2026/test/"):
        local = DATA / "test" / obj.key.removeprefix("2026/test/")
        if not obj.key.endswith("/") and not local.exists():
            local.parent.mkdir(parents=True, exist_ok=True)
            bucket.download_file(obj.key, str(local))


if AWS_ACCESS_KEY_ID:
    download_test()
    test = read_split("test")
    assert test, "no test images found, check the download"
    print(f"{len(test)} test images, scoring {MODEL_PATH}")
    print(f"Test mAP@50 (plastic_bag): {evaluate_map50(model, batches(test), DEVICE)}")